# STRAP on ADBench: ADBench protocol (disjoint train/test, GPU)

**Purpose.** Evaluate STRAP under the *same protocol as the published ADBench baselines* (Han et al., NeurIPS 2022, Table D4), so the comparison is like-for-like.

**Protocols (set `PROTOCOLS` in the config cell):**
- `adbench` (default): ADBench's own data pipeline, reproduced exactly (verified against `DataGenerator` from the ADBench repository on 21 dataset/seed pairs): datasets above 10,000 samples are subsampled to 10,000, then a stratified 70/30 split with seeds 1, 2, 3 and MinMax scaling fitted on the training part. The autoencoders are trained on the full **unlabeled** training split, which contains anomalies. Labels are never used for training.

  *Deviation from ADBench:* ADBench resamples datasets with fewer than 1,000 samples **with replacement** up to 1,000 before splitting, which places copies of the same sample in both the training and the test part (85–100% of test rows on most of these 12 datasets). This resampling is switched off (`RESAMPLE_SMALL = False`); the 12 small datasets are split as they are.
- `clean_holdout`: identical splits, but anomalies are removed from the training split.
- `clean_calib` (**default**): as `clean_holdout`, but a random 20% of the training normals is held out as a **calibration set** that the autoencoders never see. Decisions use the Box-Cox prediction-interval test with the calibration errors as reference, so each per-λ test has false positive rate α on unseen normal samples.

The 12 classical baselines are rerun on exactly the same splits and seeds (separately), so every comparison is like-for-like.

Both protocols use **disjoint** train and test sets. (The earlier clean notebook trained on all normal samples and then tested on all samples, so every normal test sample had also been used for training.)

**Decision test (all protocols):** per λ, Box-Cox is fitted on the reference errors (calibration errors for `clean_calib`, training errors otherwise), and each test error gets the one-sided prediction-interval statistic t = (y − ȳ) / (s·√(1+1/n)), compared with Student-t(n−1). Rules over the path: λ=0 only, union (≥1), ≥2, majority (≥3), all 5, and Bonferroni union (α/5). Ranking uses the maximum t over the path.

**New outputs:** empirical false positive rate on held-out normal samples, per λ and for the union, at each α; number of anomalies in the training split. The per-seed score files also store raw test errors per λ, training-error statistics, and a 5,000-sample subsample of training errors, so alternative aggregation rules, z-score baselines and normality tests can be computed later without retraining.

### Setup
1. Runtime → Change runtime type → GPU.
2. Run the setup and Drive cells.
3. The 47 datasets are downloaded automatically from the official ADBench GitHub repository (86 MB).
4. Run the remaining cells top to bottom. Results are saved after each dataset, so the run can resume after a disconnect.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
!pip install -q scipy scikit-learn pandas

import sys, os, time, warnings, zipfile, shutil, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU : {torch.cuda.get_device_name(0)}')
else:
    print('  WARNING: no GPU')

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/STRAP_ADBench'
DATA_DIR   = '/content/adbench_data'
SCORES_DIR = os.path.join(DRIVE_ROOT, 'scores')
os.makedirs(SCORES_DIR, exist_ok=True)

def results_path(protocol):
    return os.path.join(DRIVE_ROOT, f'strap_{protocol}_results.csv')
print('Results folder:', DRIVE_ROOT)

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────
# 'zip_upload' : pick ADBench_data.zip from your machine (as in the earlier notebook)
# 'zip_drive'  : copy the zip from Drive (set ZIP_ON_DRIVE)
# 'github'     : download the 47 classical .npz files from the ADBench repository
DATA_SOURCE  = 'github'        # default: official ADBench GitHub repository
ZIP_ON_DRIVE = '/content/drive/MyDrive/ADBench_data.zip'

ADBENCH_47 = ['1_ALOI','2_annthyroid','3_backdoor','4_breastw','5_campaign','6_cardio',
  '7_Cardiotocography','8_celeba','9_census','10_cover','11_donors','12_fault','13_fraud',
  '14_glass','15_Hepatitis','16_http','17_InternetAds','18_Ionosphere','19_landsat',
  '20_letter','21_Lymphography','22_magic.gamma','23_mammography','24_mnist','25_musk',
  '26_optdigits','27_PageBlocks','28_pendigits','29_Pima','30_satellite','31_satimage-2',
  '32_shuttle','33_skin','34_smtp','35_SpamBase','36_speech','37_Stamps','38_thyroid',
  '39_vertebral','40_vowels','41_Waveform','42_WBC','43_WDBC','44_Wilt','45_wine',
  '46_WPBC','47_yeast']
GITHUB_RAW = 'https://raw.githubusercontent.com/Minqi824/ADBench/main/adbench/datasets/Classical/'

def _extract_zip(zip_path):
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        for m in zf.namelist():
            if m.endswith('.npz') and os.path.basename(m):
                with zf.open(m) as src, open(os.path.join(DATA_DIR, os.path.basename(m)), 'wb') as dst:
                    shutil.copyfileobj(src, dst)

os.makedirs(DATA_DIR, exist_ok=True)
have = [f for f in os.listdir(DATA_DIR) if f.endswith('.npz')]
if DATA_SOURCE != 'github' and len(have) >= 47:
    print(f'Data already present: {len(have)} files')
elif DATA_SOURCE == 'zip_upload':
    from google.colab import files as colab_files
    up = colab_files.upload()
    _extract_zip('/content/' + list(up.keys())[0])
elif DATA_SOURCE == 'zip_drive':
    _extract_zip(ZIP_ON_DRIVE)
elif DATA_SOURCE == 'github':
    for name in ADBENCH_47:
        dst = os.path.join(DATA_DIR, name + '.npz')
        if not os.path.exists(dst):
            urllib.request.urlretrieve(GITHUB_RAW + name + '.npz', dst)

npz_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.npz'))
missing = [n for n in ADBENCH_47 if n + '.npz' not in npz_files]
tiny = [f for f in npz_files if os.path.getsize(os.path.join(DATA_DIR, f)) < 1000]
assert not missing and not tiny, f'data problem: missing={missing} tiny={tiny}'
npz_files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.npz'))
print(f'Datasets available: {len(npz_files)}')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
PROTOCOLS  = ['clean_calib']    # 'adbench', 'clean_holdout' or 'clean_calib'
CAL_FRAC   = 0.20               # clean_calib: share of training normals held out for calibration
CAL_MIN    = 20                 # ... but at least this many (or a third of the normals, if fewer)
TEST_SIZE  = 0.30               # ADBench: stratified 70/30 split
SEEDS      = [1, 2, 3]          # ADBench seeds (split and initialisation)
N_RUNS     = 3                  # all three ADBench seeds; datasets are capped at 10,000 samples
EPOCHS     = 200
LAMBDAS    = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
ALPHAS     = [0.05, 0.10]
SKIP       = []                 # dataset names to skip, e.g. ['9_census']
RESAMPLE_SMALL = False          # ADBench resamples datasets < 1,000 samples with replacement (train/test leakage)

def lambda_col(lam):
    return 'lam0' if lam == 0.0 else f'lam1e{abs(int(round(np.log10(lam))))}'
print('Protocols:', PROTOCOLS, ' seeds:', SEEDS[:N_RUNS], ' epochs:', EPOCHS)

In [ ]:
# ── STRAP core (unchanged except batched inference) ────────────────────
class TabularAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc, in_d = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(in_d, h), nn.ReLU()]; in_d = h
        enc.append(nn.Linear(in_d, latent_dim))
        self.encoder = nn.Sequential(*enc)
        dec, in_d = [], latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(in_d, h), nn.ReLU()]; in_d = h
        dec.append(nn.Linear(in_d, input_dim))
        self.decoder = nn.Sequential(*dec)
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_ae(model, X_train, lasso_lambda, epochs=200, batch_size=64, lr=1e-3, device='cpu'):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(torch.FloatTensor(X_train)), batch_size=batch_size, shuffle=True)
    model.train()
    for _ in range(epochs):
        for (xb,) in loader:
            xb = xb.to(device)
            x_hat, z = model(xb)
            loss = nn.functional.mse_loss(x_hat, xb) + lasso_lambda * z.abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def compute_mse(model, X, device='cpu', batch=65536):
    # batched to avoid GPU out-of-memory on the largest datasets (e.g. census)
    model.eval(); out = []
    for i in range(0, len(X), batch):
        x_t = torch.FloatTensor(X[i:i+batch]).to(device)
        x_hat, _ = model(x_t)
        out.append(((x_t - x_hat) ** 2).mean(dim=1).cpu().numpy())
    return np.concatenate(out)

def _grubbs_crit(n, alpha):
    t = stats.t.ppf(1.0 - alpha / (2 * n), df=n - 2)
    return ((n - 1) / np.sqrt(n)) * np.sqrt(t**2 / (n - 2 + t**2))

def grubbs_score_detect(train_err, test_err, alpha):
    N = len(train_err); N_aug = N + 1
    S1 = train_err.sum(); S2 = (train_err**2).sum()
    S1a = S1 + test_err; S2a = S2 + test_err**2
    mu = S1a / N_aug
    var = np.maximum((S2a - N_aug * mu**2) / (N_aug - 1), 0.0)
    std = np.sqrt(var)
    scores = np.where(std > 1e-12, np.abs(test_err - mu) / std, 0.0)
    return scores, scores > _grubbs_crit(N_aug, alpha)

def get_ae_config(input_dim):
    if   input_dim <= 16:  return [32, 16],       max(2, input_dim // 4)
    elif input_dim <= 64:  return [64, 32],       max(4, input_dim // 8)
    elif input_dim <= 256: return [128, 64],      16
    else:                  return [256, 128, 64], 32
print('STRAP core loaded.')

In [ ]:
# ── Data helpers, splits and metrics ───────────────────────────────────
import random
def load_dataset(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    X = d['X'].astype(np.float64); y = d['y'].astype(int).ravel()
    if len(np.unique(y)) < 2: raise ValueError('Dataset has only one class.')
    return X, y

def adbench_split(X, y, seed, test_size=TEST_SIZE, n_min=None, n_max=10000):
    """Exact replica of ADBench DataGenerator.generator (generate_duplicates=True, minmax=True)."""
    n_min = (1000 if RESAMPLE_SMALL else 0) if n_min is None else n_min
    np.random.seed(seed); random.seed(seed)
    if len(y) < n_min:
        np.random.seed(seed); random.seed(seed)
        idx = np.random.choice(np.arange(len(y)), n_min, replace=True); X, y = X[idx], y[idx]
    if len(y) > n_max:
        np.random.seed(seed); random.seed(seed)
        idx = np.random.choice(np.arange(len(y)), n_max, replace=False); X, y = X[idx], y[idx]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, shuffle=True, stratify=y)
    sc = MinMaxScaler().fit(X_tr)
    return sc.transform(X_tr), y_tr, sc.transform(X_te), y_te

def make_split(X, y, protocol, seed):
    # Returns (X_fit, X_cal or None, X_test, y_test, n_train_anomalies); data MinMax-scaled.
    # The test split is identical for all protocols.
    X_tr, y_tr, X_te, y_te = adbench_split(X, y, seed)
    f32 = lambda a: a.astype(np.float32)
    if protocol == 'adbench':          # unlabeled training split, contains anomalies
        return f32(X_tr), None, f32(X_te), y_te, int(y_tr.sum())
    X_n = X_tr[y_tr == 0]
    if protocol == 'clean_holdout':    # anomalies removed from the training split
        return f32(X_n), None, f32(X_te), y_te, 0
    if protocol == 'clean_calib':      # hold out part of the training normals for calibration
        n_cal = max(int(round(CAL_FRAC * len(X_n))), min(CAL_MIN, len(X_n) // 3))
        perm = np.random.default_rng(10_000 + seed).permutation(len(X_n))
        return f32(X_n[perm[n_cal:]]), f32(X_n[perm[:n_cal]]), f32(X_te), y_te, 0
    raise ValueError(protocol)

def safe_auroc(y, s):
    return float('nan') if len(np.unique(y)) < 2 else float(roc_auc_score(y, s))
def safe_ap(y, s):
    return float('nan') if y.sum() == 0 else float(average_precision_score(y, s))
def fpr(y, det):
    n = (y == 0).sum();  return float(det[y == 0].sum() / n) if n else float('nan')

from scipy.special import boxcox as boxcox_transform
def pi_statistic(ref_err, test_err):
    # Box-Cox prediction-interval statistic; Box-Cox fitted on the reference errors.
    # Under H0 (transformed errors normal), t ~ Student-t(n-1).
    rb, lmb = stats.boxcox(np.maximum(np.asarray(ref_err, np.float64), 1e-12))
    tb = boxcox_transform(np.maximum(np.asarray(test_err, np.float64), 1e-12), lmb)
    n = len(rb)
    return (tb - rb.mean()) / (rb.std(ddof=1) * np.sqrt(1 + 1 / n)), n - 1

def metrics_at_alpha(y, T, dfs, alpha):
    # T: (K, n_test) prediction-interval statistics; dfs: degrees of freedom per lambda
    P = np.stack([stats.t.sf(T[k], dfs[k]) for k in range(len(T))])
    D = P < alpha; votes = D.sum(axis=0); K = len(T)
    agg = T.max(axis=0)
    m = {'alpha': alpha, 'auroc': safe_auroc(y, agg), 'ap': safe_ap(y, agg)}
    rules = {'lam0': D[0], 'union': votes >= 1, 'ge2': votes >= 2, 'majority': votes >= 3,
             'all': votes == K, 'bonferroni': (P < alpha / K).any(axis=0)}
    for r, det in rules.items():
        tp = int((det & (y == 1)).sum())
        m[f'f1_{r}'] = float(f1_score(y, det, zero_division=0))
        m[f'precision_{r}'] = tp / max(1, int(det.sum())); m[f'recall_{r}'] = tp / max(1, int(y.sum()))
        m[f'fpr_{r}'] = fpr(y, det)
    for k, lam in enumerate(LAMBDAS):
        c = lambda_col(lam)
        m[f'auroc_{c}'] = safe_auroc(y, T[k]); m[f'ap_{c}'] = safe_ap(y, T[k]); m[f'fpr_test_{c}'] = fpr(y, D[k])
    return m
print('Helpers loaded.')

In [ ]:
# ── Per-dataset runner ─────────────────────────────────────────────────
def run_dataset(npz_path, protocol):
    name = os.path.splitext(os.path.basename(npz_path))[0]
    X_raw, y_raw = load_dataset(npz_path)
    input_dim = X_raw.shape[1]
    hidden_dims, latent_dim = get_ae_config(input_dim)
    print(f"\n{'='*60}\n{name}  [protocol={protocol}]  dim={input_dim}  n={len(y_raw)}")

    per_alpha = {a: [] for a in ALPHAS}
    t0 = time.time()
    for seed in SEEDS[:N_RUNS]:
        X_fit, X_cal, X_te, y_te, n_anom = make_split(X_raw, y_raw, protocol, seed)
        torch.manual_seed(seed); np.random.seed(seed)
        batch_size = min(64, max(8, len(X_fit) // 4))
        print(f'  [seed {seed}] n_train={len(X_fit)} (anomalies in train: {n_anom})  '
              f'n_cal={0 if X_cal is None else len(X_cal)}  '
              f'n_test={len(X_te)} (anomalies: {int(y_te.sum())})')
        T, dfs, train_errs, test_errs, cal_errs, tr_sub = [], [], [], [], [], []
        rng = np.random.default_rng(seed)
        for lam in LAMBDAS:
            model = train_ae(TabularAE(input_dim, hidden_dims, latent_dim), X_fit, lam,
                             epochs=EPOCHS, batch_size=batch_size, device=DEVICE)
            tr_err = compute_mse(model, X_fit, DEVICE); te_err = compute_mse(model, X_te, DEVICE)
            ref = tr_err
            if X_cal is not None:
                cal_err = compute_mse(model, X_cal, DEVICE); cal_errs.append(cal_err); ref = cal_err
            t, df = pi_statistic(ref, te_err)
            T.append(t); dfs.append(df); train_errs.append(tr_err); test_errs.append(te_err)
            # subsample of training errors for normality tests (Shapiro-Wilk needs n <= 5000)
            tr_sub.append(rng.choice(tr_err, size=min(5000, len(tr_err)), replace=False))
        T = np.stack(T); agg = T.max(axis=0)
        for a in ALPHAS:
            m = metrics_at_alpha(y_te, T, dfs, a)
            m.update({'seed': seed, 'n_train': len(X_fit), 'n_train_anomaly': n_anom,
                      'n_cal': 0 if X_cal is None else len(X_cal),
                      'n_test': len(X_te), 'n_test_anomaly': int(y_te.sum())})
            per_alpha[a].append(m)
        np.savez_compressed(os.path.join(SCORES_DIR, f'{name}_{protocol}_seed{seed}.npz'),
                            y=y_te.astype(np.int8), agg_scores=agg.astype(np.float32),
                            per_lambda_scores=T.astype(np.float32),
                            per_lambda_cal_err=(np.stack(cal_errs).astype(np.float32) if cal_errs else np.zeros((0,))),
                            # raw errors and training-error statistics, so alternative
                            # aggregation rules and z-score baselines can be computed offline
                            per_lambda_test_err=np.stack(test_errs).astype(np.float32),
                            train_err_mean=np.array([t.mean() for t in train_errs], dtype=np.float64),
                            train_err_std=np.array([t.std(ddof=1) for t in train_errs], dtype=np.float64),
                            train_err_n=np.array([len(t) for t in train_errs], dtype=np.int64),
                            train_err_sample=np.stack(tr_sub).astype(np.float32),
                            lambdas=np.array(LAMBDAS, dtype=np.float32))
        m05 = per_alpha[ALPHAS[0]][-1]
        print(f"    AUROC={m05['auroc']:.4f}  AP={m05['ap']:.4f}  @alpha={ALPHAS[0]}: "
              f"FPR(lam0)={m05['fpr_lam0']:.3f}  FPR(union)={m05['fpr_union']:.3f}  F1(majority)={m05['f1_majority']:.3f}")

    rows = []
    for a, runs in per_alpha.items():
        row = {'dataset': name, 'protocol': protocol, 'input_dim': input_dim,
               'n_runs': len(runs), 'elapsed_s': round(time.time() - t0, 1), 'status': 'ok'}
        for k in runs[0]:
            v = [r[k] for r in runs]
            row[k] = float(np.mean(v)) if k not in ('alpha',) else a
            if k not in ('alpha', 'seed'): row[f'{k}_std'] = float(np.std(v))
        rows.append(row)
    return rows
print('Runner loaded.')

In [ ]:
# ── One-time cleanup: drop small-dataset results made with resampling ──
# Runs once per protocol (a marker file prevents repeats). Only the 12 datasets below are
# affected by RESAMPLE_SMALL; their old results are removed so they rerun without leakage.
SMALL = ['4_breastw', '14_glass', '15_Hepatitis', '18_Ionosphere', '21_Lymphography', '29_Pima',
         '37_Stamps', '39_vertebral', '42_WBC', '43_WDBC', '45_wine', '46_WPBC']
for protocol in PROTOCOLS:
    marker = os.path.join(DRIVE_ROOT, f'.small_rerun_{protocol}')
    if RESAMPLE_SMALL or os.path.exists(marker): continue
    p = results_path(protocol)
    if os.path.exists(p):
        df = pd.read_csv(p); df[~df['dataset'].isin(SMALL)].to_csv(p, index=False)
    for f in os.listdir(SCORES_DIR):
        if any(f.startswith(f'{s}_{protocol}_seed') for s in SMALL): os.remove(os.path.join(SCORES_DIR, f))
    open(marker, 'w').write('done'); print(f'{protocol}: small-dataset results cleared')

# ── Main loop (resumable) ──────────────────────────────────────────────
def done_datasets(path):
    if not os.path.exists(path): return set()
    df = pd.read_csv(path); return set(df.loc[df['status'] == 'ok', 'dataset'])

for protocol in PROTOCOLS:
    path = results_path(protocol); done = done_datasets(path)
    todo = sorted([f for f in npz_files if os.path.splitext(f)[0] not in done | set(SKIP)],
                  key=lambda f: os.path.getsize(os.path.join(DATA_DIR, f)))
    print(f'\n### protocol={protocol}: {len(done)} done, {len(todo)} to run')
    for i, f in enumerate(todo):
        ds = os.path.splitext(f)[0]; print(f'[{i+1}/{len(todo)}] {ds}')
        try:
            rows = run_dataset(os.path.join(DATA_DIR, f), protocol)
        except Exception as e:
            print('  [ERROR]', e)
            rows = [{'dataset': ds, 'protocol': protocol, 'status': 'error', 'error_msg': str(e)}]
        pd.DataFrame(rows).to_csv(path, mode='a', header=not os.path.exists(path), index=False)
print('\nFinished.')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────
RULE_NAMES = ['lam0', 'union', 'ge2', 'majority', 'all', 'bonferroni']
for protocol in PROTOCOLS:
    df = pd.read_csv(results_path(protocol)); ok = df[df['status'] == 'ok']
    print(f'\n=== protocol={protocol}: {ok.dataset.nunique()} datasets ===')
    s = ok[ok['alpha'] == ALPHAS[0]]
    print(f'Ranking (max t over the path): AUROC {s.auroc.mean():.4f}  AP {s.ap.mean():.4f}')
    for a in ALPHAS:
        s = ok[ok['alpha'] == a]
        print(f'alpha={a}:')
        for r in RULE_NAMES:
            print(f'   {r:10s} F1 {s[f"f1_{r}"].mean():.4f}  precision {s[f"precision_{r}"].mean():.4f}  '
                  f'recall {s[f"recall_{r}"].mean():.4f}  FPR {s[f"fpr_{r}"].mean():.4f}')
        print('   per-lambda FPR:', '  '.join(f"{lambda_col(l)}={s[f'fpr_test_{lambda_col(l)}'].mean():.4f}" for l in LAMBDAS))
